*© Sabrine Bendimerad · [sbendimerad.github.io](https://sbendimerad.github.io) · Licence [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/): you may share and adapt this material with credit, for non-commercial use, under the same licence. Datasets keep their own licences.*

# Supervised Machine Learning - Naive Bayes

<!-- TOC START min:2 max:4 link:true asterisk:true update:true -->
* [What you will learn in this class](#what-you-will-learn-in-this-class)
* [Bayes theorem](#bayes-theorem)
* [Naive Bayes](#naive-bayes)
  * [Case where $X_i$ is qualitative](#case-where--is-qualitative)
  * [Case where $X_i$ is quantitative](#case-where--is-quantitative)
* [General remarks](#general-remarks)
<!-- TOC END -->

In [ ]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
# colour palette of the course
pio.templates["course"] = go.layout.Template(
    layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"]
)
pio.templates.default = "plotly_white+course"

## What you will learn in this class

This course is about the so-called Naive Bayes model. This model assumes that the explanatory variables are independent of each other (given the target): a very strong hypothesis, which is very rarely true in practice. Nevertheless, this model can be very useful, and it gives a good view of the influence of each variable on the target variable.

In statistics, Naive Bayes classifiers belong to the family of probabilistic classifiers based on Bayes' theorem.

## Conditional probability
Let's start this lesson with the concept of conditional probability. Think of the following example:

You flip two balanced coins, one after the other. It may not seem so, but we have already used many probabilistic concepts here:
* Flipping a coin defines a *random variable*: a set of possible values (here *tail* and *head*), each with an associated probability (or a probability distribution when the set of values is continuous, or more precisely uncountable). Here the coins are balanced, which means $P(tail) = P(head) = 0.5$
* Flipping two coins one after the other introduces two more concepts. First, a *random experiment*: flipping the two coins and looking at the outcome of the two random variables. Second, *independence*: the two coins are not related to each other in any way, so the outcome of one flip has no influence on the other flip, and vice versa.

Let us now define two *events*:
* Event $A$: the first coin is *tail*
* Event $B$: both coins are *tail*

We can now calculate the probability of each event:

$P(A)=P(coin_1 = tail)=0.5$

$P(B)=P(coin_1 = tail,\, coin_2 = tail)=P(coin_1 = tail)\times P(coin_2 = tail) = 0.5\times 0.5 = 0.25$

To calculate the probability of event B, we are actually calculating the *joint* probability that coin 1 and coin 2 are both tail in the same random experiment. For two independent events, the joint probability equals the product of the individual probabilities.

:::tip Intuition
A conditional probability is a probability computed in a *smaller world*. To get $P(B|A)$, keep only the experiments where $A$ happened, forget all the others, and count how often $B$ happens among the ones that remain. Here: among the flips where coin 1 is tail, how often is coin 2 also tail?
:::

Let us now introduce the concept of conditional probability. The conditional probability of $B$ given $A$ is the probability that event $B$ happens, given that event $A$ has already happened. It is written as follows:

$$
\begin{aligned}
P(B|A) &= P(coin_1 = tail,\, coin_2 = tail|coin_1 = tail) \\
       &= P(coin_1 = tail | coin_1 = tail)\times P(coin_2 = tail | coin_1 = tail) \\
       &= 1 \times P(coin_2 = tail) \\
       &= 1 \times 0.5 \\
       &= 0.5
\end{aligned}
$$

To go from line 1 to line 2, we use the fact that coin 1 and coin 2 are independent: the joint conditional probability becomes the product of the two conditional probabilities. The probability that coin 1 is tail given that coin 1 is tail is... 1! We already know that coin 1 is tail here. Then, because coin 1 does not influence coin 2, the second conditional probability is simply the probability that coin 2 is tail: 0.5. So the conditional probability of $B$ given $A$ is 0.5, whereas the probability of $B$ alone is 0.25!

The rest of this lesson is based on these probability rules.

**See it with data 🔬 Flipping two coins 10,000 times**

Instead of trusting the formulas, let's flip two coins 10,000 times with numpy and simply count. $P(B|A)$ is computed exactly as in the intuition box: keep only the experiments where coin 1 is tail, then count how often both coins are tail.

In [ ]:
rng = np.random.default_rng(0)
coins = rng.choice(["head", "tail"], size=(10_000, 2))   # one row = one experiment (coin 1, coin 2)

A = coins[:, 0] == "tail"                                # event A: coin 1 is tail
B = (coins[:, 0] == "tail") & (coins[:, 1] == "tail")    # event B: both coins are tail

print(f"P(A)   ≈ {A.mean():.3f}   (theory: 0.5)")
print(f"P(B)   ≈ {B.mean():.3f}   (theory: 0.25)")
print(f"P(B|A) ≈ {B[A].mean():.3f}   (theory: 0.5)  <- B counted only among the {A.sum()} experiments where A happened")

P(A)   ≈ 0.501   (theory: 0.5)
P(B)   ≈ 0.245   (theory: 0.25)
P(B|A) ≈ 0.489   (theory: 0.5)  <- B counted only among the 5007 experiments where A happened


The counts agree with the theory: $P(A) \approx 0.501$, $P(B) \approx 0.245$ and $P(B|A) \approx 0.489$. Knowing that coin 1 is tail *doubles* the chance that both coins are tail, from about 0.25 to about 0.5. The small gaps with the theory are just random noise: with more flips, they would shrink.

## Bayes theorem

:::tip Intuition
Bayes' theorem lets you *reverse* a conditional probability. What we often know is $P(B|A)$: a lab knows how often its test is positive for a sick person. What we want is $P(A|B)$: a patient with a positive test wants to know whether they are sick. The formula starts from $P(A)$, what we believed *before* seeing $B$ (the "prior"), and multiplies it by $\frac{P(B|A)}{P(B)}$: how much more likely $B$ is when $A$ is true than in general.
:::

Bayes theorem corresponds to the following statement:

Let $A$ and $B$ be two events (with $P(B) > 0$). Then the following equality holds:

### $P(A|B)=\frac{P(B|A)\cdot{P(A)}}{P(B)}$


The conditional probability of $A$ given $B$, $P(A|B)$, equals the conditional probability of $B$ given $A$, $P(B|A)$, times the probability of $A$, divided by the probability of $B$.

**An example.** A disease affects 1% of the population. A test is positive for 90% of the sick people, but also for 5% of the healthy ones. You test positive: what is the probability that you are sick? Bayes' theorem gives $P(\text{sick}|+)=\frac{0.9\times0.01}{0.9\times0.01+0.05\times0.99}\approx15\%$. Counting people makes it easy to see why:

Bayes' theorem with counts: out of 10,000 people, 90 sick and 495 healthy people test positive

**See it with data 🔬 The medical test on 100,000 people**

15% sounds surprisingly low for a test that detects 90% of sick people. Let's simulate a population of 100,000 people, with 1% of sick people, give everyone the test, and look only at the people who test positive.

In [ ]:
rng = np.random.default_rng(0)
n = 100_000
sick = rng.random(n) < 0.01                                   # 1% of the population is sick
positive = np.where(sick, rng.random(n) < 0.90,               # positive for 90% of the sick...
                    rng.random(n) < 0.05)                     # ...and for 5% of the healthy

print(f"sick people: {sick.sum()}, positive tests: {positive.sum()}")
print(f"positive AND sick: {(sick & positive).sum()}, positive AND healthy: {(~sick & positive).sum()}")
print(f"P(sick | positive) ≈ {sick[positive].mean():.3f}")

sick people: 987, positive tests: 5883
positive AND sick: 885, positive AND healthy: 4998
P(sick | positive) ≈ 0.150


Out of 100,000 people, 987 are sick and 5,883 test positive, but only 885 of those positive tests come from sick people: $P(\text{sick}|+) \approx 0.150$, the 15% given by Bayes' theorem. The test is good, but healthy people are so numerous that their 5% of false alarms (about 5,000 people) outnumber the true positives. That is the role of the prior $P(A)$ in the formula.

## Naive Bayes

:::tip Intuition
For each class, Naive Bayes builds a simple *profile*: how common the class is, and what each feature usually looks like in this class. To classify a new observation, it asks, for each class, "how well does this observation fit the profile?", weighs the answer by how common the class is, and picks the best class.
:::

Let $Y$ be the qualitative target variable we are trying to predict (so this is a classification problem), and $X=(X_1,X_2,...,X_p)$ a collection of explanatory variables. For each observation, we want to estimate the distribution $P(Y|X)$: the probability that $Y$ takes each of its possible values, given the values of $X$ for this observation.

This is where Bayes' theorem comes in:


### $P(Y|X)=\frac{P(X|Y)\cdot{P(Y)}}{P(X)}$



The denominator does not involve $Y$, so it has no influence on the model's results. We only need the numerator, which can be decomposed step by step using the properties of conditional probabilities. The property we use here is:
$P(A,B)=P(A|B)P(B)$
We then get the following:

### $P(X|Y)\cdot{P(Y)}=P(X_1,X_2,...,X_p,Y)$

### $P(X|Y)\cdot{P(Y)}=P(X_1|X_2,...,X_p,Y)\cdot{P(X_2,...,X_p,Y)}$

### $P(X|Y)\cdot{P(Y)}=P(X_1|X_2,...,X_p,Y)\cdot{P(X_2|X_3,...,X_p,Y)}\cdot{P(X_3,...,X_p,Y)}$

### $P(X|Y)\cdot{P(Y)}=P(X_1|X_2,...,X_p,Y)\cdot{P(X_2|X_3,...,X_p,Y)}\cdot{P(X_3|X_4,...,X_p,Y)...P(X_p|Y)\cdot{P(Y)}}$

:::tip Intuition
Think of a doctor who looks at each symptom *separately*: "a high glucose is typical of diabetics, so is a high BMI, so is an older age". Each clue multiplies the odds on its own, and the doctor never asks whether two clues say the same thing. That is the naive assumption.
:::

This is where we need the fundamental (and naive) assumption that allows us to build our estimates: *the explanatory variables are independent of each other, given $Y$*. Then, for all $i$ between $1$ and $p$:

### $P(X_i|X_{i+1},...,X_p,Y)=P(X_i|Y)$


So you get:

### $P(Y|X)=\frac{P(Y)P(X_1|Y)P(X_2|Y)...P(X_p|Y)}{P(X)}$



This is very simple to calculate, since we just need to estimate the distribution of each $X_i$ for each value of $Y$.



### Case where $X_i$ is qualitative

If $X_i$ is a qualitative explanatory variable that takes the categories $x_{i1},...,x_{iq}$, then we can write:

### $\hat{P}(X_i=x_{ik}|Y=y)=\frac{Card(X_i=x_{ik}, Y=y)}{Card(Y=y)}$


We estimate the probability that $X_i$ takes the category $x_{ik}$, given that $Y = y$, as the proportion of observations where $X_i=x_{ik}$ among all the observations where $Y = y$.
This is very easy to calculate, and can be very accurate even with very little data.

**See it with data 🔬 Naive Bayes by hand on 14 patients**

Let's apply the formulas on a tiny table built from the course data (`diabetes.csv`): 14 patients (8 without and 6 with diabetes), and three qualitative features made from their measurements: glucose (high if ≥ 140), BMI (obese if ≥ 30) and age (40+ or under 40).

In [ ]:
diabetes = pd.read_csv("data/diabetes.csv")
patients = diabetes[diabetes["BMI"] > 0]                      # a BMI of 0 means "missing"
small = pd.concat([patients[patients["Outcome"] == 0].sample(8, random_state=1),
                   patients[patients["Outcome"] == 1].sample(6, random_state=1)])

table = pd.DataFrame({
    "glucose": np.where(small["Glucose"] >= 140, "high", "normal"),
    "bmi": np.where(small["BMI"] >= 30, "obese", "not obese"),
    "age": np.where(small["Age"] >= 40, "40+", "under 40"),
    "diabetes": np.where(small["Outcome"] == 1, "yes", "no"),
})
table

,glucose,bmi,age,diabetes
0,normal,obese,under 40,no
1,normal,not obese,under 40,no
2,normal,obese,under 40,no
3,normal,not obese,40+,no
4,normal,obese,40+,no
5,normal,not obese,40+,no
6,high,obese,under 40,no
7,normal,not obese,40+,no
8,normal,not obese,40+,yes
9,high,obese,40+,yes


A new patient arrives with a high glucose, an obese BMI and an age of 40+. For each class $y$, we compute $P(Y=y)\times P(X_1|Y=y)\times P(X_2|Y=y)\times P(X_3|Y=y)$, each probability being a simple proportion in the table. Dividing by the sum of the two scores plays the role of $P(X)$.

In [ ]:
prior = table["diabetes"].value_counts(normalize=True)        # P(Y)
new = {"glucose": "high", "bmi": "obese", "age": "40+"}

scores = {}
for y in ["no", "yes"]:
    rows = table[table["diabetes"] == y]
    scores[y] = prior[y]
    for feature, value in new.items():
        p = (rows[feature] == value).mean()                   # P(X_i = value | Y = y)
        print(f"P({feature} = {value} | diabetes = {y}) = {p:.3f}")
        scores[y] *= p
    print(f"P(diabetes = {y}) = {prior[y]:.3f}  ->  score = {scores[y]:.4f}\n")

total = sum(scores.values())                                  # plays the role of P(X)
for y in scores:
    print(f"P(diabetes = {y} | new patient) = {scores[y] / total:.3f}")

P(glucose = high | diabetes = no) = 0.125
P(bmi = obese | diabetes = no) = 0.500
P(age = 40+ | diabetes = no) = 0.500
P(diabetes = no) = 0.571  ->  score = 0.0179

P(glucose = high | diabetes = yes) = 0.333
P(bmi = obese | diabetes = yes) = 0.833
P(age = 40+ | diabetes = yes) = 0.833
P(diabetes = yes) = 0.429  ->  score = 0.0992

P(diabetes = no | new patient) = 0.153
P(diabetes = yes | new patient) = 0.847


Now the same thing with scikit-learn's `CategoricalNB`. By default it adds 1 to every count (*Laplace smoothing*, `alpha=1`), so that a category never seen in a class does not get a probability of exactly 0, which would wipe out the whole product. Here we set `alpha` almost to 0, to reproduce our raw counts.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.naive_bayes import CategoricalNB

features = ["glucose", "bmi", "age"]
encoder = OrdinalEncoder()                                    # categories -> 0, 1, ...
model = CategoricalNB(alpha=1e-10).fit(encoder.fit_transform(table[features]), table["diabetes"])

X_new = encoder.transform(pd.DataFrame([new]))
for y, p in zip(model.classes_, model.predict_proba(X_new)[0]):
    print(f"CategoricalNB: P(diabetes = {y} | new patient) = {p:.3f}")

CategoricalNB: P(diabetes = no | new patient) = 0.153
CategoricalNB: P(diabetes = yes | new patient) = 0.847


By hand, the score for "yes" is $\frac{6}{14}\times\frac{2}{6}\times\frac{5}{6}\times\frac{5}{6} \approx 0.0992$ and the score for "no" is $\frac{8}{14}\times\frac{1}{8}\times\frac{4}{8}\times\frac{4}{8} \approx 0.0179$. After normalization, the new patient has diabetes with probability 0.847, and `CategoricalNB` gives exactly the same 0.847. Naive Bayes is nothing more than these few proportions and one product: the "training" is just counting.

### Case where $X_i$ is quantitative

When $X_i$ is quantitative, we generally go back to the qualitative case by cutting the range of values of $X_i$ into $K$ intervals, indexed by $k\in{[[1,K]]}$ and delimited by the values $-\infty=\alpha_0,\alpha_1,...,\alpha_{k-1},+\infty=\alpha_k$. The probability distribution becomes:

### $\hat{P}(X_i=x_{ik}\in[\alpha_j, \alpha_{j+1}]|Y=y)=\frac{Card(X_i\in[\alpha_ j,\alpha_{j+1}],Y=y)}{Card(Y=y)}$


I.e. the proportion of observations where the value of $X_i$ belongs to the interval $[\alpha_j,\alpha_{j+1}]$, among all the observations where $Y = y$. This technique is called the discretization of a continuous variable.

Another way to estimate the distribution of $X_i$ given $Y$ is to assume that $P(X_i|Y)$ follows a normal distribution, whose parameters $\mu_i$ (mean) and $\sigma_i$ (standard deviation) are estimated from the data available on $X_i$. Under this normality assumption, $P(X_i|Y)$ follows a normal distribution with parameters:


### $\mu_{iy}=\frac{1}{N_y}\sum_{j=1}^{N_y}x_{ij}$ 


This is the average value of $X_i$ among the $N_y$ individuals for whom $Y = y$. Similarly, we calculate the variance of the normal distribution:


### $\sigma_{iy}^2=\frac{1}{N_y-1}\sum_{j=1}^{N_y}(x_{ij}-\mu_{iy})^2$



This is the estimated variance of $X_i$ among the individuals for whom ***Y = y***. Once these parameters are estimated, we get:


:::tip Intuition
Instead of counting, we draw one bell curve per class, centred on the class mean and as wide as the class standard deviation. The *height* of the curve at a value tells how typical this value is for the class. (Strictly speaking it is a density, not a probability, but since we only compare classes with each other, it plays the same role.)
:::

### $\hat{P}(X_i=x_{ik}|Y=y)=\frac{1}{\sqrt{2\pi\sigma_{iy}^2}}exp(\frac{-(x_{ik}-\mu_{iy})^2}{2\sigma_{iy}^2})$


Once all the conditional probabilities have been calculated, we get, for each observation and for each category of $Y$, a probability that determines our classification. Each observation is assigned to the category of $Y$ that is the most probable, given the values of the explanatory variables $X$.

**See it with data 🔬 One bell curve per class**

Let's look at the glucose of the 763 patients of `diabetes.csv` whose glucose is known. For each class we estimate $\mu$ and $\sigma$, then draw the normal curve on top of the real histogram.

In [ ]:
glucose = diabetes[diabetes["Glucose"] > 0]                   # a glucose of 0 means "missing"
params = glucose.groupby("Outcome")["Glucose"].agg(["mean", "std"])
print(params.round(1))

x = np.linspace(40, 220, 300)
fig = go.Figure()
for outcome, name, color in [(0, "no diabetes", "#3B47C4"), (1, "diabetes", "#E4572E")]:
    mu, sigma = params.loc[outcome]
    fig.add_trace(go.Histogram(x=glucose.loc[glucose["Outcome"] == outcome, "Glucose"], histnorm="probability density",
                               name=f"{name} (data)", marker_color=color, opacity=0.3, xbins=dict(size=5)))
    fig.add_trace(go.Scatter(x=x, y=np.exp(-(x - mu) ** 2 / (2 * sigma ** 2)) / np.sqrt(2 * np.pi * sigma ** 2),
                             mode="lines", name=f"{name} (normal curve)", line=dict(color=color, width=3)))
fig.add_vline(x=150, line_dash="dash", line_color="#111111")
fig.update_layout(barmode="overlay", title="Glucose given the class: data and normal curves",
                  xaxis_title="glucose (mg/dL)", yaxis_title="density")
fig.show()

          mean   std
Outcome             
0        110.6  24.8
1        142.3  29.6


In [ ]:
def normal_density(x, mu, sigma):                             # the formula above
    return np.exp(-(x - mu) ** 2 / (2 * sigma ** 2)) / np.sqrt(2 * np.pi * sigma ** 2)

prior = glucose["Outcome"].value_counts(normalize=True)
density = {outcome: normal_density(150, *params.loc[outcome]) for outcome in (0, 1)}
for outcome in (0, 1):
    print(f"Outcome = {outcome}: P(glucose = 150 | Outcome) ≈ {density[outcome]:.4f}, P(Outcome) = {prior[outcome]:.3f}")
posterior = density[1] * prior[1] / (density[1] * prior[1] + density[0] * prior[0])
print(f"P(diabetes | glucose = 150) ≈ {posterior:.3f}")

Outcome = 0: P(glucose = 150 | Outcome) ≈ 0.0046, P(Outcome) = 0.651
Outcome = 1: P(glucose = 150 | Outcome) ≈ 0.0130, P(Outcome) = 0.349
P(diabetes | glucose = 150) ≈ 0.605


The two classes have different curves: mean 110.6 (σ = 24.8) without diabetes and 142.3 (σ = 29.6) with diabetes. At a glucose of 150 (dashed line), the red curve is about 2.9 times higher than the blue one (0.0130 vs 0.0046). But diabetics are only 35% of the patients, so after weighing by the priors, $P(\text{diabetes}|\text{glucose}=150) \approx 0.605$. The curves also show the limits of the assumption: the real histograms are not perfect bell curves, but they are close enough to be useful.

## General remarks

One advantage of the Naive Bayes model is that, if we transform the explanatory variables into qualitative variables, we don't need any assumption about their distributions. However, the fundamental hypothesis that the explanatory variables are independent is very rarely true in practice.

**See it with data 🔬 Naive Bayes vs logistic regression**

The independence assumption is false on almost every real dataset. Does it make Naive Bayes useless? Let's compare `GaussianNB` with a logistic regression on the 8 features of `diabetes.csv`, on the same train/test split. Each model is trained 20 times to measure an average fit time.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = diabetes.drop(columns="Outcome"), diabetes["Outcome"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)

models = {"Gaussian Naive Bayes": GaussianNB(),
          "logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))}
for name, model in models.items():
    start = time.perf_counter()
    for _ in range(20):
        model.fit(X_train, y_train)
    fit_ms = (time.perf_counter() - start) / 20 * 1000
    print(f"{name:22s} test accuracy = {model.score(X_test, y_test):.3f}   fit time ≈ {fit_ms:.1f} ms")

Gaussian Naive Bayes   test accuracy = 0.766   fit time ≈ 1.7 ms


logistic regression    test accuracy = 0.781   fit time ≈ 6.9 ms


Naive Bayes reaches a test accuracy of 0.766, close to the 0.781 of the logistic regression, and it fits faster (compare the training times printed above; the exact values depend on your computer). Fitting a Gaussian Naive Bayes only means computing a mean and a standard deviation per feature and per class. Even with a wrong assumption, it is a strong and very cheap baseline.

**See it with data 🔬 When the independence assumption breaks**

What if two features say *exactly* the same thing? Let's train Naive Bayes on Glucose, BMI and Age, then again after adding 5 copies of Glucose. The copies bring no new information, but Naive Bayes believes they are 5 independent clues, and multiplies the glucose evidence 6 times.

In [ ]:
from sklearn.metrics import accuracy_score, log_loss

cols = ["Glucose", "BMI", "Age"]
X_train_dup, X_test_dup = X_train[cols].copy(), X_test[cols].copy()
for k in range(1, 6):
    X_train_dup[f"Glucose_copy{k}"] = X_train_dup["Glucose"]
    X_test_dup[f"Glucose_copy{k}"] = X_test_dup["Glucose"]

proba = {"3 features": GaussianNB().fit(X_train[cols], y_train).predict_proba(X_test[cols])[:, 1],
         "Glucose copied 5 times": GaussianNB().fit(X_train_dup, y_train).predict_proba(X_test_dup)[:, 1]}
for name, p in proba.items():
    extreme = ((p < 0.05) | (p > 0.95)).mean()
    print(f"{name:23s} accuracy = {accuracy_score(y_test, p > 0.5):.3f}   log loss = {log_loss(y_test, p):.3f}"
          f"   predictions below 5% or above 95%: {extreme:.0%}")

3 features              accuracy = 0.786   log loss = 0.454   predictions below 5% or above 95%: 8%
Glucose copied 5 times  accuracy = 0.771   log loss = 0.992   predictions below 5% or above 95%: 78%


In [ ]:
fig = go.Figure()
for name, p in proba.items():
    fig.add_trace(go.Histogram(x=p, name=name, opacity=0.6, xbins=dict(start=0, end=1, size=0.05)))
fig.update_layout(barmode="overlay", title="Predicted probabilities of diabetes on the test set",
                  xaxis_title="predicted probability of diabetes", yaxis_title="number of test patients")
fig.show()

The accuracy barely moves (0.786 → 0.771), but the probabilities change a lot: with the copies, 78% of the predictions are below 5% or above 95%, against 8% before. The model has become over-confident, and the log loss (which punishes confident wrong answers; lower is better) more than doubles, from 0.454 to 0.992. Lesson: with Naive Bayes, the predicted *class* is often fine, but don't trust the predicted *probabilities* when features are strongly correlated, and remove redundant features.

Naive Bayes models can be aggregated, just like the decision trees of a random forest. This generally gives much more stable results. And if, as in a random forest, each model is built on only part of the explanatory variables, the independence hypothesis is better respected.